# Fig 2 — E2 vs E3 playhead-gap boxplot (with scatter overlay)

Headline figure. Shows that naive switching's playhead gap scales
with filterDelay while aligned switching is bounded by 1 GOP.
Individual run dots overlaid so the (very tight) within-cell spread
is visible.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "figures"))

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from _data import load_aggregate
from _style import (
    COL_ALIGNED,
    COL_NAIVE,
    GOP_DURATION_MS,
    apply_acm_style,
    fig_one_col,
)

apply_acm_style()

In [ ]:
e2 = load_aggregate("e2").assign(mode="naive")
e3 = load_aggregate("e3").assign(mode="aligned")
combined = pd.concat([e2, e3], ignore_index=True)
combined["offset_s"] = combined["cell_id"].str.extract(r"offset(\d+)").astype(int)

In [ ]:
fig, ax = fig_one_col(height_in=2.6)

offsets = sorted(combined["offset_s"].unique())
positions_naive = [i - 0.20 for i in range(len(offsets))]
positions_aligned = [i + 0.20 for i in range(len(offsets))]

naive_data = [combined.query("mode == 'naive' and offset_s == @o")["max_playhead_gap_ms"].values for o in offsets]
aligned_data = [combined.query("mode == 'aligned' and offset_s == @o")["max_playhead_gap_ms"].values for o in offsets]

bp_n = ax.boxplot(
    naive_data, positions=positions_naive, widths=0.32, patch_artist=True,
    medianprops={"color": "black", "linewidth": 0.7},
    whiskerprops={"linewidth": 0.5}, capprops={"linewidth": 0.5},
    flierprops={"marker": "", "markersize": 0},  # we draw our own dots
    whis=(0, 100),  # whiskers reach extremes
)
bp_a = ax.boxplot(
    aligned_data, positions=positions_aligned, widths=0.32, patch_artist=True,
    medianprops={"color": "black", "linewidth": 0.7},
    whiskerprops={"linewidth": 0.5}, capprops={"linewidth": 0.5},
    flierprops={"marker": "", "markersize": 0},
    whis=(0, 100),
)
for patch in bp_n["boxes"]:
    patch.set_facecolor(COL_NAIVE)
    patch.set_edgecolor(COL_NAIVE)
    patch.set_alpha(0.55)
for patch in bp_a["boxes"]:
    patch.set_facecolor(COL_ALIGNED)
    patch.set_edgecolor(COL_ALIGNED)
    patch.set_alpha(0.55)

# Overlay individual run dots so the within-cell distribution is visible
# even when the boxes collapse on a log scale.
rng = np.random.default_rng(seed=0)
for pos, vals in zip(positions_naive, naive_data):
    jitter = rng.uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(pos + jitter, vals, s=8, color=COL_NAIVE,
               edgecolor="black", linewidth=0.4, zorder=3, alpha=0.95)
for pos, vals in zip(positions_aligned, aligned_data):
    jitter = rng.uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(pos + jitter, vals, s=8, color=COL_ALIGNED,
               edgecolor="black", linewidth=0.4, zorder=3, alpha=0.95)

ax.set_yscale("log")
ax.set_ylim(500, 60000)  # leave room above largest naive (~35k) and below 1 GOP
ax.set_xticks(range(len(offsets)))
ax.set_xticklabels([f"{o}" for o in offsets])
ax.set_xlabel("Filter delay (s)")
ax.set_ylabel("max |playhead gap| (ms, log scale)")

ax.axhline(GOP_DURATION_MS, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
ax.text(len(offsets) - 0.5, GOP_DURATION_MS * 1.12, "1 GOP",
        fontsize=7, ha="right", va="bottom")

ax.legend(handles=[
    mpatches.Patch(facecolor=COL_NAIVE, alpha=0.55, edgecolor=COL_NAIVE, label="naive (E2)"),
    mpatches.Patch(facecolor=COL_ALIGNED, alpha=0.55, edgecolor=COL_ALIGNED, label="aligned (E3)"),
], loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False, fontsize=7)

In [ ]:
fig.savefig(Path.cwd().parent / "figures" / "fig2_e2_e3_playhead_gap.pdf")
fig.savefig(Path.cwd().parent / "figures" / "fig2_e2_e3_playhead_gap.png", dpi=200)